# Day 15 — Solution: Fama–MacBeth

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from scipy import stats
DATA_SOURCE = os.environ.get("QRC_DATA", "real")

def ols_multi(X_cols, y):
    X = np.column_stack([np.ones(len(y))] + [np.asarray(c, float) for c in X_cols])
    y = np.asarray(y, float)
    n, k = X.shape
    XtX_inv = np.linalg.inv(X.T @ X)
    b = XtX_inv @ X.T @ y
    e = y - X @ b
    s2 = e @ e / (n - k)
    se = np.sqrt(np.diag(s2 * XtX_inv))
    r2 = 1 - (e @ e) / ((y - y.mean()) @ (y - y.mean()))
    return {"b": b, "se": se, "r2": r2, "resid": e}

## E1 — the worked miniature

In [ ]:
z = pd.DataFrame({"A": [ 1.2,  0.4, -1.5, -0.1],
                  "B": [ 0.3, -1.1,  0.7,  0.1],
                  "C": [-0.8,  0.2,  0.4,  0.2]}, index=["n1","n2","n3","n4"])   # columns = months
r_next = pd.DataFrame({"A": [0.012, -0.028, 0.004, -0.017],
                       "B": [0.031,  0.006, -0.011, 0.009],
                       "C": [-0.008,  0.021, 0.016, -0.002]}, index=["n1","n2","n3","n4"])

lam = {m: ols_multi([z[m].values], r_next[m].values)["b"][1] for m in z.columns}
lam = pd.Series(lam)
fm_mean, fm_se = lam.mean(), lam.std(ddof=1) / np.sqrt(len(lam))
print(f"monthly slopes: {lam.round(4).to_dict()}")
print(f"FM: mean {fm_mean:+.4f}  SE {fm_se:.4f}  t {fm_mean/fm_se:+.2f}")

zpool = pd.concat([z[c] for c in z.columns]).values
rpool = pd.concat([r_next[c] for c in r_next.columns]).values
pooled = ols_multi([zpool], rpool)
print(f"pooled: slope {pooled['b'][1]:+.4f}  classical SE {pooled['se'][1]:.4f}  t {pooled['b'][1]/pooled['se'][1]:+.2f}")

**Expected reasoning.** Monthly slopes: A ≈ −0.3bp, B ≈ −2.3bp,
C ≈ +18.9bp → FM mean ≈ +5.4bp, SE ≈ 6.7bp (std 11.6/√3), t ≈ +0.8;
pooled: slope ≈ +1.7bp, classical SE ≈ 6.9bp, t ≈ +0.25. Three things
to notice in this toy. (1) The FM mean is the EQUAL-weighted average of
monthly slopes while the pooled slope variance-weights months (say the
difference out loud — they need not and do not coincide: +5.4bp vs
+1.7bp here; equal weighting is a CHOICE Fama–MacBeth makes, and it is
the right one when each month is one draw of the premium). (2) The two
SEs nearly coincide here — of course: three independent toy months have
no common component, so the disease FM treats hasn't been injected yet;
E3's tournament injects it (common factor × persistent characteristic)
and watches the pooled SE break. (3) With 4 names the numbers are toys,
but the arithmetic of step 2 (mean of slopes, std/√T) is exactly the
arithmetic used at FF92's scale. Everything else this week is this cell
scaled by thousands of months × names and audited for real.

## E2 — the machine

In [ ]:
def fama_macbeth(frame, ycol, xcols):
    rows, r2s = {}, {}
    for m, g in frame.groupby(level=0):
        g = g[[ycol] + xcols].dropna()
        if len(g) < 10:
            continue
        Zs = [(g[c] - g[c].mean()) / g[c].std() for c in xcols]
        f = ols_multi(Zs, g[ycol].values)
        rows[m] = f["b"]; r2s[m] = f["r2"]
    lam = pd.DataFrame(rows, index=["const"] + xcols).T
    return lam, pd.Series(r2s)

def fm_table(lam, r2=None):
    T = len(lam)
    out = pd.DataFrame({"mean": lam.mean(),
                        "SE": lam.std(ddof=1) / np.sqrt(T),
                        "share_pos": (lam > 0).mean()})
    out["t"] = out["mean"] / out["SE"]
    if r2 is not None:
        out["mean_monthly_R2"] = r2.mean() if len(out) == 1 else np.nan
    return out

**Expected reasoning.** The machine is 12 lines because the idea is 12
lines: groupby month, z-score within month, small OLS, collect. **Guard
the two invariants inside it forever:** z-scoring happens *within* month
(cross-sectional!), and cross-sections with < 10 names are skipped rather
than silently fit on 3 points — at panel scale both bugs masquerade as
results. `fm_table` follows the literature's rows: mean (the premium),
SE/t (its significance), share_pos (steadiness), mean monthly R²
(explanatory size; 0.02–0.05 is the normal band in this course's worlds).

## E3 — the null tournament: pooled vs FM

In [ ]:
n_runs, N, T = 300, 50, 120
rej_pooled, rej_fm = 0, 0
for i in range(n_runs):
    rng = np.random.default_rng(30000 + i)
    m_t = rng.normal(0, 1, T)
    betas = rng.uniform(0.5, 1.5, N)
    R = np.outer(m_t, betas) + rng.normal(0, 1.0, (T, N))     # month-corr across names ~0.5
    z = rng.normal(0, 1, N)                                   # PERSISTENT per name, true effect ZERO
    f = ols_multi([np.tile(z, T)], R.T.ravel())
    rej_pooled += abs(f["b"][1] / f["se"][1]) > 1.96
    # FM
    lams = np.array([ols_multi([z], R[t])["b"][1] for t in range(T)])
    rej_fm += abs(lams.mean() / (lams.std(ddof=1) / np.sqrt(T))) > 1.96
print(f"rejection of TRUE null at 5%: pooled classical {rej_pooled/n_runs:.3f} | FM {rej_fm/n_runs:.3f}")

**Expected numbers.** Pooled classical rejects the TRUE null at **~35–
40%** — a 5% test firing seven times too often (SEs understated ~2–3×);
FM holds ≈ **0.03–0.06**. This is the concepts-sheet's last row, earned:
on panels, *pooled classical SEs are not a rounding error, they are a
broken test at 5% nominal* — and the Fama–MacBeth average over monthly
cross-sections is the 50-year-old, still-standard repair.

**The wrinkle's mechanism (important).** Redraw z fresh EVERY month and
the pooled failure mostly vanishes: the score z·u only inherits the
common-factor correlation when z has cross-sectional persistence
(E[z_i z_j] ≠ 0). Real characteristics — size, BM, vol levels — persist
for years, which is exactly why the pooled failure is the default and
not the edge case. **The premium estimate itself (≈ 0) was never wrong;
only its certification was.**

## E4 — FM on your panel

In [ ]:
from qrc.data import get_prices
from qrc.synth import synthetic_returns, synthetic_prices
from qrc.universe import load_universe

if DATA_SOURCE == "real":
    names = load_universe("research50")[:50]
    px = get_prices(names, start="2010-01-01")
    vol_sh = get_prices(names, start="2010-01-01", field="volume")
    rets_d = np.log(px).diff()
    dv_d = px.ffill() * vol_sh
else:
    names = [f"S{i}" for i in range(50)]
    rets_d = synthetic_returns(n_days=3600, n_assets=50, seed=8, corr=0.4,
                               drift_spread=0.0002)
    rets_d.columns = names
    rng = np.random.default_rng(8)
    adv = np.exp(rng.normal(np.log(1e8), 0.8, len(names)))
    dv_d = pd.DataFrame(synthetic_prices(n_days=3600, n_assets=50, seed=8, corr=0.4).values
                        * (adv * np.exp(rng.normal(0, 0.4, rets_d.shape))),
                        index=rets_d.index, columns=names)

per = rets_d.index.to_period("M")
r_m = rets_d.groupby(per).sum(); vol_m = rets_d.groupby(per).std() * np.sqrt(21)
dv_m = dv_d.groupby(per).median()
for dfm in (r_m, vol_m, dv_m):
    dfm.index = dfm.index.to_timestamp()

frame = pd.DataFrame({"vol": vol_m.stack(), "ldv": np.log(dv_m).stack()})
frame["r_next"] = r_m.stack().groupby(level=1).shift(-1)
frame = frame.dropna(); frame.index.names = ["month", "name"]

if DATA_SOURCE == "synthetic":      # the plant: -20bp per cross-sectional sd of vol; dv: nothing
    zv_cs = frame.groupby(level=0)["vol"].transform(lambda s: (s - s.mean()) / s.std())
    frame["r_next"] = frame["r_next"] - 0.002 * zv_cs

lam, r2 = fama_macbeth(frame, "r_next", ["vol", "ldv"])
tab = fm_table(lam, r2)
print((tab.assign(mean_bp=tab["mean"] * 1e4, SE_bp=tab["SE"] * 1e4)[["mean_bp","SE_bp","t","share_pos"]]).round(2).to_string())
print(f"\nmean monthly R2: {r2.mean():.3f}")
acf = [lam["vol"].autocorr(l) for l in range(1, 6)]
print("ACF of lam_vol:", [f"{a:+.2f}" for a in acf])

**Expected pattern (synthetic, measured).** λ̂_vol ≈ −20 to −25bp/σ,
t ≈ −3 — the plant recovered within ~1 SE of truth; λ̂_dv ≈ 0 (t ≈ +0.5)
— null correctly unrejected. share_pos(vol) ≈ 0.44 (negative-premium
months a bit more common, as planted). λ̂_t ACF lags 1–5 within ±0.1 →
the simple std/√T SE is adequate; were it 0.3+, the fix is the one you
derived in the hints: intercept-regression of λ̂_t with HAC SEs. Mean
monthly R² ≈ 0.065 (two z-scored characteristics, one true — the plant's
contribution plus the ~2/49 k/N fog floor). **Real mode:** vol typically
negative-ish, weak in megacaps (compare day 14's time-series slope —
and see E5); interpretation must carry the research50 survivorship
sentence.

## E5 — the two slopes (exemplar)

Mechanism for disagreement: **mean reversion within name vs premium
across names.** If each stock's OWN high-vol months are followed by
normal-vol rebounds (vol mean-reverts idiosyncratically fast), the
time-series slope can be ≈ 0 everywhere — while stocks that are
CHRONICALLY high-vol earn less than chronically calm ones in every month
(a cross-sectional compensation story: lottery demand / leverage
constraints). Same word "volatility", two non-interchangeable objects:
within-name variation vs between-name level. FF92's tables are
cross-sectional (step 1 monthly across stocks) — the premium question is
"what does the market pay across names", which is why BM/size slopes,
not reversion dynamics, populate them. **Summary you should be able to
recite: time-series slope prices variation; cross-sectional slope prices
exposure.** Day 19 puts both on the same panel and reads them side by
side.